In [4]:
# ============================================================
# CELL 1 — SETUP, LOAD, AND FIX NaN
#   Added SimpleImputer to fill NaN values with median
#   This fixes the ValueError in Cells 3, 4, 5
#   VLF (100% NaN) gets filled with 0 → correctly dropped
#   LF/HF (3.9% NaN) get filled with median → usable
# ============================================================

import numpy as np
import pandas as pd
import os
import sys
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from scipy.stats import spearmanr

COMPONENT_PATH = r"C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals"
sys.path.insert(0, os.path.join(COMPONENT_PATH, "src"))
from utils.config import RESULTS_PATH

# ── Load feature matrix ──────────────────────────────────────
csv_path    = os.path.join(
    RESULTS_PATH, "objective1", "wesad_features.csv"
)
features_df = pd.read_csv(csv_path)

feature_cols = [c for c in features_df.columns
                if c not in ['label', 'subject_id']]

X        = features_df[feature_cols].values
y        = features_df['label'].values
subjects = features_df['subject_id'].values

print("=" * 60)
print("FEATURE SELECTION — SETUP")
print("=" * 60)
print(f"Feature matrix loaded: {features_df.shape}")
print(f"Feature columns: {len(feature_cols)}")
print(f"Subjects: {sorted(features_df['subject_id'].unique())}")

print()
print("LABEL DISTRIBUTION:")
for lid, name in {
    0:'Normal', 1:'Pre-stress', 2:'Stress'
}.items():
    count = (y == lid).sum()
    pct   = count / len(y) * 100
    print(f"  Class {lid} ({name:10s}): {count:4d} ({pct:.1f}%)")

# ── Fix NaN values ───────────────────────────────────────────
# Why: VLF=100% NaN, LF/HF=3.9% NaN
# sklearn models cannot handle NaN
# Solution: fill with column median
# VLF gets median=0 → all same value → selected by 0 methods
# → correctly dropped by voting step
print()
print("NaN CHECK AND FIX:")
nan_counts = pd.DataFrame(X, columns=feature_cols).isna().sum()
nan_feats  = nan_counts[nan_counts > 0]

if len(nan_feats) > 0:
    print(f"  Features with NaN:")
    for feat, count in nan_feats.items():
        pct = count / len(X) * 100
        print(f"    {feat:30s}: {count} ({pct:.1f}%)")

    print(f"\n  Filling with column median...")
    imputer = SimpleImputer(strategy='median')
    X_clean = imputer.fit_transform(X)
    print(f"  NaN after fill: {np.isnan(X_clean).sum()}")
else:
    X_clean = X.copy()
    print(f"  No NaN values found")

print()
print("FEATURE LIST:")
for i, feat in enumerate(feature_cols):
    has_nan = nan_counts.get(feat, 0) > 0
    flag    = " ← had NaN, filled" if has_nan else ""
    print(f"  {i+1:2d}. {feat}{flag}")

print()
print("Cell 1 complete — X_clean ready for feature selection")

FEATURE SELECTION — SETUP
Feature matrix loaded: (883, 42)
Feature columns: 40
Subjects: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]

LABEL DISTRIBUTION:
  Class 0 (Normal    ):  587 (66.5%)
  Class 1 (Pre-stress):   29 (3.3%)
  Class 2 (Stress    ):  267 (30.2%)

NaN CHECK AND FIX:
  Features with NaN:
    VLF                           : 883 (100.0%)

  Filling with column median...
  NaN after fill: 0

FEATURE LIST:
   1. SDNN
   2. RMSSD
   3. AVNN
   4. MeanHR
   5. SDHR
   6. pNN50
   7. pNN20
   8. LF
   9. HF
  10. LF_HF
  11. VLF ← had NaN, filled
  12. LFnu
  13. HFnu
  14. SD1
  15. SD2
  16. SCL_mean
  17. SCL_std
  18. SCL_slope
  19. EDA_mean
  20. EDA_std
  21. EDA_range
  22. EDA_slope
  23. SCR_count
  24. SCR_amp_mean
  25. SCR_width_mean
  26. EDA_q10
  27. EDA_q25
  28. EDA_q75
  29. EDA_q90
  30. deriv1_mean
  31. deriv1_std
  32. deriv2_mean
  33. wavelet_approx_mean
  34. wavelet_approx_energy
  35. wavelet_d1_energy
  36. wavelet_d2_energy
  37. wavelet_

In [6]:
# ============================================================
# CELL 2 — METHOD 1: SPEARMAN CORRELATION
#
# What it does:
#   Measures monotonic relationship between each feature
#   and the stress label (0, 1, 2)
#   Higher absolute correlation = more stress-related
#
# Why Spearman not Pearson:
#   Pearson assumes linear relationship
#   Stress features often have nonlinear relationships
#   Spearman works on ranks — more robust
#
# Papers: 10, 12 used correlation-based selection
# ============================================================

print("=" * 60)
print("METHOD 1 — SPEARMAN CORRELATION WITH STRESS LABEL")
print("=" * 60)

spearman_scores = {}

for feat in feature_cols:
    vals = features_df[feat].values
    # Spearman correlation with ordinal stress label
    corr, pval = spearmanr(vals, y)
    spearman_scores[feat] = abs(corr)  # absolute value

# Sort by correlation strength
spearman_sorted = sorted(
    spearman_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

print(f"\n{'Feature':30s} {'Correlation':12s} {'Selected':8s}")
print("-" * 55)

# Select features with correlation > 0.1
spearman_threshold  = 0.1
spearman_selected   = []

for feat, score in spearman_sorted:
    selected = score >= spearman_threshold
    flag     = "YES" if selected else "no"
    print(f"  {feat:28s} {score:.4f}       {flag}")
    if selected:
        spearman_selected.append(feat)

print(f"\nSpearman selected: {len(spearman_selected)} features")
print(f"  {spearman_selected}")

METHOD 1 — SPEARMAN CORRELATION WITH STRESS LABEL

Feature                        Correlation  Selected
-------------------------------------------------------
  AVNN                         0.6246       YES
  MeanHR                       0.6246       YES
  SDHR                         0.4786       YES
  VLF                          nan       no
  EDA_q90                      0.4242       YES
  EDA_q75                      0.4231       YES
  SCL_mean                     0.4230       YES
  wavelet_approx_energy        0.4230       YES
  EDA_mean                     0.4230       YES
  wavelet_approx_mean          0.4230       YES
  EDA_q25                      0.4222       YES
  EDA_q10                      0.4213       YES
  SCL_slope                    0.3641       YES
  EDA_slope                    0.3631       YES
  SCR_count                    0.3599       YES
  pNN20                        0.3572       YES
  SCR_amp_mean                 0.3480       YES
  EDA_std                   

In [7]:
# ============================================================
# CELL 3 — METHOD 2: EXTRA TREES FEATURE IMPORTANCE
# uses X_clean (NaN filled) instead of X
# Error was: ValueError: Input X contains NaN
# Solution:  SimpleImputer in Cell 1 filled all NaN
#            X_clean has no NaN values
#            et_model.fit(X_clean, y) works correctly
# ============================================================

from sklearn.utils.class_weight import compute_class_weight

print("=" * 60)
print("METHOD 2 — EXTRA TREES FEATURE IMPORTANCE")
print("=" * 60)

classes   = np.array([0, 1, 2])
class_wts = compute_class_weight(
    'balanced', classes=classes, y=y
)
class_weight_dict = {
    0: class_wts[0],
    1: class_wts[1],
    2: class_wts[2]
}
print(f"Class weights: {class_weight_dict}")

# Using X_clean — no NaN values
et_model = ExtraTreesClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
et_model.fit(X_clean, y)

et_importances = dict(zip(
    feature_cols,
    et_model.feature_importances_
))
et_sorted = sorted(
    et_importances.items(),
    key=lambda x: x[1],
    reverse=True
)

print(f"\n{'Feature':30s} {'Importance':12s} {'Selected':8s}")
print("-" * 55)

mean_importance = np.mean(list(et_importances.values()))
et_selected     = []

for feat, score in et_sorted:
    selected = score >= mean_importance
    flag     = "YES" if selected else "no"
    print(f"  {feat:28s} {score:.4f}       {flag}")
    if selected:
        et_selected.append(feat)

print(f"\nExtra Trees selected: {len(et_selected)} features")
print(f"  Threshold = mean importance = {mean_importance:.4f}")
print(f"  {et_selected}")

METHOD 2 — EXTRA TREES FEATURE IMPORTANCE
Class weights: {0: 0.501419647927314, 1: 10.149425287356323, 2: 1.1023720349563046}

Feature                        Importance   Selected
-------------------------------------------------------
  AVNN                         0.0902       YES
  MeanHR                       0.0870       YES
  EDA_slope                    0.0457       YES
  wavelet_d3_energy            0.0364       YES
  deriv1_std                   0.0322       YES
  deriv2_mean                  0.0307       YES
  pNN20                        0.0296       YES
  SDHR                         0.0286       YES
  SCR_count                    0.0284       YES
  SCR_width_mean               0.0281       YES
  EDA_q75                      0.0265       YES
  SD2                          0.0256       no
  SCL_slope                    0.0253       no
  pNN50                        0.0250       no
  EDA_q25                      0.0244       no
  EDA_q10                      0.0242       no
 

In [8]:
# ============================================================
# CELL 4 — METHOD 3: RECURSIVE FEATURE ELIMINATION
#
# Fix applied: uses X_clean instead of X
# Same NaN issue would have occurred here too
# ============================================================

from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

print("=" * 60)
print("METHOD 3 — RECURSIVE FEATURE ELIMINATION (RFE)")
print("=" * 60)

n_features_rfe = 20

rf_estimator = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rfe = RFE(
    estimator=rf_estimator,
    n_features_to_select=n_features_rfe,
    step=1
)

print(f"Running RFE — selecting top {n_features_rfe} features...")
print("Takes 2-3 minutes...")

# Using X_clean — no NaN values
rfe.fit(X_clean, y)

rfe_mask     = rfe.support_
rfe_selected = [
    feat for feat, sel in zip(feature_cols, rfe_mask) if sel
]
rfe_ranking  = dict(zip(feature_cols, rfe.ranking_))

rfe_sorted = sorted(
    rfe_ranking.items(),
    key=lambda x: x[1]
)

print(f"\n{'Feature':30s} {'Ranking':10s} {'Selected':8s}")
print("-" * 52)

for feat, rank in rfe_sorted:
    selected = feat in rfe_selected
    flag     = "YES" if selected else "no"
    print(f"  {feat:28s} {rank:8d}       {flag}")

print(f"\nRFE selected: {len(rfe_selected)} features")
print(f"  {rfe_selected}")

METHOD 3 — RECURSIVE FEATURE ELIMINATION (RFE)
Running RFE — selecting top 20 features...
Takes 2-3 minutes...

Feature                        Ranking    Selected
----------------------------------------------------
  AVNN                                1       YES
  MeanHR                              1       YES
  SDHR                                1       YES
  SD1                                 1       YES
  SD2                                 1       YES
  SCL_mean                            1       YES
  EDA_mean                            1       YES
  EDA_std                             1       YES
  EDA_range                           1       YES
  SCR_count                           1       YES
  SCR_amp_mean                        1       YES
  SCR_width_mean                      1       YES
  EDA_q10                             1       YES
  EDA_q75                             1       YES
  EDA_q90                             1       YES
  deriv2_mean                     

In [11]:
# ============================================================
# CELL 5 — METHOD 4: PERMUTATION IMPORTANCE — FIXED THRESHOLD
#
# Fix applied: changed threshold from 0.0 to strictly > 0
# Previous version selected 39/40 features because
# threshold=0.0 kept everything with non-negative score
# Most features scored exactly 0.000 → should be excluded
# New threshold: only features with score > 0.0 kept
# ============================================================

print("=" * 60)
print("METHOD 4 — PERMUTATION IMPORTANCE (Relief substitute)")
print("=" * 60)

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

print("Running permutation importance...")
print("This takes 3-5 minutes...")

rf_perm = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_perm.fit(X_clean, y)

perm_result = permutation_importance(
    rf_perm, X_clean, y,
    n_repeats=30,       # increased from 10 for more stability
    random_state=42,
    n_jobs=-1
)

perm_scores = dict(zip(
    feature_cols,
    perm_result.importances_mean
))
perm_sorted = sorted(
    perm_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

print(f"\n{'Feature':30s} {'Score':12s} {'Std':10s} {'Selected':8s}")
print("-" * 65)

# Fix: threshold = strictly positive (score > 0)
# Previous bug: threshold=0.0 selected features with score=0
# A score of exactly 0 means the feature contributes nothing
# A negative score means the feature actively hurts accuracy
perm_threshold  = 0.0
relief_selected = []

perm_stds = dict(zip(
    feature_cols,
    perm_result.importances_std
))

for feat, score in perm_sorted:
    std      = perm_stds.get(feat, 0)
    selected = score > perm_threshold  # strictly greater than 0
    flag     = "YES" if selected else "no"
    print(f"  {feat:28s} {score:10.6f}   {std:8.6f}   {flag}")
    if selected:
        relief_selected.append(feat)

print(f"\nPermutation importance selected: "
      f"{len(relief_selected)} features")
print(f"  Threshold = strictly > {perm_threshold}")
print(f"  Features scoring exactly 0.0 = contribute nothing = dropped")
print(f"  {relief_selected}")

METHOD 4 — PERMUTATION IMPORTANCE (Relief substitute)
Running permutation importance...
This takes 3-5 minutes...

Feature                        Score        Std        Selected
-----------------------------------------------------------------
  AVNN                           0.027709   0.004006   YES
  MeanHR                         0.011325   0.002632   YES
  SCL_mean                       0.004417   0.002012   YES
  SCR_amp_mean                   0.003662   0.001453   YES
  ACC_mean                       0.001321   0.001058   YES
  ACC_std                        0.000868   0.000811   YES
  EDA_mean                       0.000831   0.000712   YES
  SCR_count                      0.000566   0.000566   YES
  SDHR                           0.000264   0.000479   YES
  wavelet_d3_energy              0.000189   0.000422   YES
  SDNN                           0.000000   0.000000   no
  RMSSD                          0.000000   0.000000   no
  pNN50                          0.000000   0.000

In [12]:
# ============================================================
# CELL 6 — COMBINE ALL 4 METHODS
#  uses results from Cells 2-5
# ============================================================

print("=" * 60)
print("COMBINING ALL 4 METHODS")
print("=" * 60)

method_names   = ['Spearman', 'ExtraTrees', 'RFE', 'Relief']
method_results = [
    spearman_selected,
    et_selected,
    rfe_selected,
    relief_selected
]

vote_counts = {}
for feat in feature_cols:
    votes = sum(
        1 for method in method_results if feat in method
    )
    vote_counts[feat] = votes

votes_sorted = sorted(
    vote_counts.items(),
    key=lambda x: (x[1], et_importances.get(x[0], 0)),
    reverse=True
)

print(f"\n{'Feature':30s} {'Spearman':10s} {'ExtraTrees':12s} "
      f"{'RFE':6s} {'Relief':8s} {'Votes':6s} {'Keep':6s}")
print("-" * 80)

final_selected = []
min_votes      = 2

for feat, votes in votes_sorted:
    s1 = "YES" if feat in spearman_selected else "no"
    s2 = "YES" if feat in et_selected       else "no"
    s3 = "YES" if feat in rfe_selected      else "no"
    s4 = "YES" if feat in relief_selected   else "no"

    keep = votes >= min_votes
    flag = "KEEP" if keep else "drop"

    print(f"  {feat:28s} {s1:8s}   {s2:10s}   "
          f"{s3:4s}   {s4:6s}   {votes:4d}   {flag}")

    if keep:
        final_selected.append(feat)

print(f"\n{'='*60}")
print(f"FINAL SELECTED FEATURES: {len(final_selected)}")
print(f"{'='*60}")
for i, feat in enumerate(final_selected):
    print(f"  {i+1:2d}. {feat}")

print(f"\nDropped features: "
      f"{len(feature_cols) - len(final_selected)}")
dropped = [f for f in feature_cols if f not in final_selected]
for feat in dropped:
    print(f"  - {feat}  (votes: {vote_counts[feat]})")

COMBINING ALL 4 METHODS

Feature                        Spearman   ExtraTrees   RFE    Relief   Votes  Keep  
--------------------------------------------------------------------------------
  AVNN                         YES        YES          YES    YES         4   KEEP
  MeanHR                       YES        YES          YES    YES         4   KEEP
  wavelet_d3_energy            YES        YES          YES    YES         4   KEEP
  SDHR                         YES        YES          YES    YES         4   KEEP
  SCR_count                    YES        YES          YES    YES         4   KEEP
  SCR_width_mean               YES        YES          YES    no          3   KEEP
  EDA_q75                      YES        YES          YES    no          3   KEEP
  SCR_amp_mean                 YES        no           YES    YES         3   KEEP
  EDA_mean                     YES        no           YES    YES         3   KEEP
  SCL_mean                     YES        no           YES    

In [13]:
# ============================================================
# CELL 7 — SAVE SELECTED FEATURE MATRIX
# ============================================================

print("=" * 60)
print("SAVING SELECTED FEATURE MATRIX")
print("=" * 60)

selected_df = features_df[
    final_selected + ['label', 'subject_id']
].copy()

print(f"Original shape: {features_df.shape}")
print(f"Selected shape: {selected_df.shape}")
print(f"Features reduced: "
      f"{len(feature_cols)} → {len(final_selected)}")

save_path = os.path.join(
    RESULTS_PATH, "objective1",
    "wesad_features_selected.csv"
)
selected_df.to_csv(save_path, index=False)

size_mb = os.path.getsize(save_path) / 1024 / 1024
print(f"\nSaved: {save_path}")
print(f"Size:  {size_mb:.2f} MB")

summary_rows = []
for feat, votes in vote_counts.items():
    summary_rows.append({
        'feature':       feat,
        'votes':         votes,
        'selected':      feat in final_selected,
        'spearman':      feat in spearman_selected,
        'extratrees':    feat in et_selected,
        'rfe':           feat in rfe_selected,
        'relief':        feat in relief_selected,
        'et_importance': et_importances.get(feat, 0),
        'spearman_corr': spearman_scores.get(feat, 0)
    })

summary_df = pd.DataFrame(summary_rows).sort_values(
    'votes', ascending=False
)

summary_path = os.path.join(
    RESULTS_PATH, "objective1",
    "feature_selection_summary.csv"
)
summary_df.to_csv(summary_path, index=False)
print(f"Summary saved: {summary_path}")

print(f"\nFEATURE SELECTION COMPLETE")
print(f"Ready for model training")
print(f"\nFinal feature set ({len(final_selected)} features):")
for i, feat in enumerate(final_selected):
    et_imp  = et_importances.get(feat, 0)
    sp_corr = spearman_scores.get(feat, 0)
    print(f"  {i+1:2d}. {feat:30s} "
          f"ET={et_imp:.4f}  Spearman={sp_corr:.4f}")

SAVING SELECTED FEATURE MATRIX
Original shape: (883, 42)
Selected shape: (883, 24)
Features reduced: 40 → 22

Saved: C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals\results\objective1\wesad_features_selected.csv
Size:  0.34 MB
Summary saved: C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals\results\objective1\feature_selection_summary.csv

FEATURE SELECTION COMPLETE
Ready for model training

Final feature set (22 features):
   1. AVNN                           ET=0.0902  Spearman=0.6246
   2. MeanHR                         ET=0.0870  Spearman=0.6246
   3. wavelet_d3_energy              ET=0.0364  Spearman=0.2025
   4. SDHR                           ET=0.0286  Spearman=0.4786
   5. SCR_count                      ET=0.0284  Spearman=0.3599
   6. SCR_width_mean                 ET=0.0281  Spearman=0.2605
   7. EDA_q75                        ET=0.0265  Spearman=0.4231
   8. SCR_amp_mean                   ET=0.0220  Sp